# Hooks de Middleware en LangChain

**Prerrequisito:** Haber revisado `skills_demo.ipynb`, donde se introduce `AgentMiddleware` con el patrón `wrap_model_call`.

---

## ¿Qué es un hook?

Un **hook** es un punto de intercepción en el ciclo de vida del agente donde puedes ejecutar lógica personalizada. LangChain define dos familias de hooks:

| Familia | Hooks disponibles | ¿Cuándo se usa? |
|---------|-------------------|------------------|
| **Node-style** | `before_agent`, `before_model`, `after_model`, `after_agent` | Logging, validación, actualización de estado |
| **Wrap-style** | `wrap_model_call`, `wrap_tool_call` | Retries, caché, transformación de request/response |

### Ciclo de ejecución con middlewares

```
┌─────────────────────────────────────────────────────────────┐
│  before_agent()         ← node-style, una vez por invocación │
│                                                              │
│  ┌── Loop del agente ───────────────────────────────────┐   │
│  │  before_model()      ← node-style, antes de cada LLM │   │
│  │                                                       │   │
│  │  wrap_model_call() → [ wrap_tool_call() → tool ] → LLM   │
│  │                                                       │   │
│  │  after_model()       ← node-style, después de cada LLM│  │
│  └───────────────────────────────────────────────────────┘   │
│                                                              │
│  after_agent()          ← node-style, una vez al finalizar   │
└─────────────────────────────────────────────────────────────┘
```

> **Regla de orden con múltiples middlewares:**
> - `before_*` → de primero a último en la lista
> - `after_*` → de último a primero (inverso)
> - `wrap_*` → anidado (el primero de la lista envuelve a todos los demás)

## Setup

In [ ]:
# Instalar dependencias si es necesario
# !pip install -q -U langchain langchain-openai langchain-core langgraph python-dotenv

In [ ]:
import getpass
import os

from dotenv import load_dotenv

load_dotenv(".env")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Modelo: {OPENAI_MODEL}")

---

## Hook 1 — `before_model` + `after_model` (node-style)

Los hooks **node-style** se definen como métodos en una clase `AgentMiddleware`. Reciben el estado completo del agente (`AgentState`) y el objeto `Runtime`. Retornan un `dict` con actualizaciones de estado (o `None` para no modificar nada).

Este primer ejemplo implementa **logging estructurado**:
- `before_model` → registra cuántos mensajes hay en contexto antes de llamar al LLM
- `after_model` → registra el contenido de la respuesta y los tokens consumidos

Para que funcione async (el agente en el API usa `ainvoke`), también se implementan `abefore_model` y `aafter_model`.

In [ ]:
import time
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime


class LoggingMiddleware(AgentMiddleware):
    """Logging estructurado de cada llamada al modelo.

    Hooks utilizados:
      - before_model / abefore_model  → registra inicio de llamada
      - after_model  / aafter_model   → registra respuesta y tokens
    """

    # Variable de instancia para medir tiempo entre before y after
    _start_time: float = 0.0

    # ── Hooks SYNC ──────────────────────────────────────────────────────────

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        self._start_time = time.perf_counter()
        msg_count = len(state["messages"])
        print(f"[LoggingMiddleware] ▶ before_model | mensajes en contexto: {msg_count}")
        return None  # sin modificaciones al estado

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        elapsed_ms = (time.perf_counter() - self._start_time) * 1000
        last_msg = state["messages"][-1]
        tokens = getattr(last_msg, "usage_metadata", {}) or {}
        print(
            f"[LoggingMiddleware] ◀ after_model  | "
            f"tiempo: {elapsed_ms:.0f}ms | "
            f"tokens entrada: {tokens.get('input_tokens', '?')} | "
            f"tokens salida: {tokens.get('output_tokens', '?')}"
        )
        return None

    # ── Hooks ASYNC (para uso con ainvoke) ──────────────────────────────────

    async def abefore_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        self._start_time = time.perf_counter()
        msg_count = len(state["messages"])
        print(f"[LoggingMiddleware] ▶ abefore_model | mensajes en contexto: {msg_count}")
        return None

    async def aafter_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        elapsed_ms = (time.perf_counter() - self._start_time) * 1000
        last_msg = state["messages"][-1]
        tokens = getattr(last_msg, "usage_metadata", {}) or {}
        print(
            f"[LoggingMiddleware] ◀ aafter_model  | "
            f"tiempo: {elapsed_ms:.0f}ms | "
            f"tokens entrada: {tokens.get('input_tokens', '?')} | "
            f"tokens salida: {tokens.get('output_tokens', '?')}"
        )
        return None


print("LoggingMiddleware definido")

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=OPENAI_MODEL)

agent_logging = create_agent(
    model=llm,
    system_prompt="Eres un asistente útil. Responde en español y de forma concisa.",
    middleware=[LoggingMiddleware()],
)

# Invocación — observa los mensajes impresos por el middleware
response = agent_logging.invoke(
    {"messages": [HumanMessage(content="¿Cuál es la capital de Francia?")]}
)

print("\nRespuesta:", response["messages"][-1].content)

---

## Hook 2 — `after_model` con `jump_to` (safety guard)

Un hook node-style puede **modificar el flujo de ejecución** retornando un dict con la clave `jump_to`. Para habilitar esta capacidad, el hook debe declararse con `@hook_config(can_jump_to=[...])` (o bien el decorador `@after_model` con el mismo argumento).

**Destinos de salto disponibles:**
- `'end'` → termina la ejecución del agente (va directo al `after_agent`)
- `'tools'` → salta al nodo de herramientas
- `'model'` → regresa al nodo del modelo

En este ejemplo implementamos un **guardia de contenido** que detecta términos prohibidos en la respuesta del LLM y la reemplaza por un mensaje de advertencia, cortocircuitando cualquier llamada adicional.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime


class SafetyGuardMiddleware(AgentMiddleware):
    """Detecta términos prohibidos en la respuesta y corta la ejecución.

    Hook utilizado:
      - after_model con can_jump_to=["end"]
    """

    DEFAULT_BLOCKED_TERMS = ["BLOQUEADO", "PELIGROSO", "ILEGAL"]

    def __init__(self, blocked_terms: list[str] | None = None) -> None:
        super().__init__()
        self.blocked_terms = [
            t.upper() for t in (blocked_terms or self.DEFAULT_BLOCKED_TERMS)
        ]

    @hook_config(can_jump_to=["end"])
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_message = state["messages"][-1]
        content = str(getattr(last_message, "content", "")).upper()

        for term in self.blocked_terms:
            if term in content:
                print(f"[SafetyGuard] 🚫 Término bloqueado detectado: '{term}'. Cortando ejecución.")
                return {
                    "messages": [
                        AIMessage(
                            content="Lo siento, no puedo responder a esa consulta."
                        )
                    ],
                    "jump_to": "end",  # ← sale del loop del agente
                }

        print(f"[SafetyGuard]  Respuesta limpia.")
        return None  # flujo normal


print("SafetyGuardMiddleware definido ✓")

In [ ]:
# Para demostrar el safety guard, creamos una herramienta que devuelve
# una cadena con un término bloqueado, simulando una respuesta 'peligrosa'
from langchain.tools import tool


@tool
def consulta_contenido(tema: str) -> str:
    """Consulta información sobre un tema. Devuelve contenido sobre el tema solicitado."""
    # En este demo, simula una respuesta con término bloqueado para
    # un tema específico
    if "hack" in tema.lower():
        return "BLOQUEADO: Este contenido ha sido marcado como no permitido."
    return f"Información sobre {tema}: contenido seguro y relevante."


agent_safety = create_agent(
    model=llm,
    system_prompt=(
        "Eres un asistente. Cuando el usuario pregunte sobre un tema, "
        "usa la herramienta consulta_contenido para obtener información."
    ),
    tools=[consulta_contenido],
    middleware=[SafetyGuardMiddleware()],
)

print("--- Pregunta segura ---")
resp_ok = agent_safety.invoke(
    {"messages": [HumanMessage(content="Cuéntame sobre Python como lenguaje de programación")]}
)
print("Respuesta:", resp_ok["messages"][-1].content)

print("\n--- Pregunta que activa el guard ---")
resp_blocked = agent_safety.invoke(
    {"messages": [HumanMessage(content="Enséñame técnicas de hacking")]}
)
print("Respuesta:", resp_blocked["messages"][-1].content)

---

## Hook 3 — Custom `state_schema` (contador de tokens con presupuesto)

Los middlewares pueden **extender el estado del agente** con campos personalizados usando `state_schema`. El estado extendido persiste durante toda la invocación, permitiendo:
- Compartir datos entre hooks (e.g., pasar un valor de `before_model` a `after_model`)
- Acumular métricas a lo largo de múltiples iteraciones del loop
- Tomar decisiones basadas en historial de la invocación actual

En este ejemplo implementamos un **presupuesto de tokens**: si el agente consume más tokens de los permitidos, se interrumpe la ejecución.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langgraph.runtime import Runtime
from typing_extensions import NotRequired


# 1. Extender el estado del agente con nuestro campo personalizado
class TokenBudgetState(AgentState):
    """Estado extendido con contador acumulado de tokens."""
    token_count: NotRequired[int]  # tokens acumulados en esta invocación


class TokenBudgetMiddleware(AgentMiddleware):
    """Lleva un contador de tokens y corta la ejecución si supera el presupuesto.

    Hooks utilizados:
      - before_model: verifica si el presupuesto está agotado → jump_to 'end'
      - after_model:  acumula tokens de la última respuesta en el estado

    Atributos de clase:
      - state_schema: indica al agente que use TokenBudgetState en lugar de AgentState
    """

    state_schema = TokenBudgetState  # ← registra el schema extendido

    def __init__(self, budget: int = 1000) -> None:
        super().__init__()
        self.budget = budget

    # ── VERIFICA antes de llamar al modelo ──────────────────────────────────

    @hook_config(can_jump_to=["end"])
    def before_model(
        self, state: TokenBudgetState, runtime: Runtime
    ) -> dict[str, Any] | None:
        current = state.get("token_count", 0)
        print(f"[TokenBudget] before_model | tokens consumidos: {current}/{self.budget}")

        if current >= self.budget:
            print(f"[TokenBudget]  Presupuesto agotado. Cortando ejecución.")
            return {
                "messages": [
                    AIMessage(content="He alcanzado el límite de procesamiento para esta consulta.")
                ],
                "jump_to": "end",
            }
        return None

    # ── ACUMULA tokens después de la respuesta del modelo ───────────────────

    def after_model(
        self, state: TokenBudgetState, runtime: Runtime
    ) -> dict[str, Any] | None:
        last_msg = state["messages"][-1]
        usage = getattr(last_msg, "usage_metadata", None) or {}
        new_tokens = usage.get("input_tokens", 0) + usage.get("output_tokens", 0)
        accumulated = state.get("token_count", 0) + new_tokens
        print(f"[TokenBudget] after_model  | +{new_tokens} tokens → total: {accumulated}")
        # Retornar el nuevo valor del campo personalizado
        return {"token_count": accumulated}


print("TokenBudgetMiddleware definido ✓")

In [ ]:
# Agente con presupuesto generoso (no se agota en este ejemplo)
agent_budget = create_agent(
    model=llm,
    system_prompt="Eres un asistente conciso. Responde en español.",
    middleware=[TokenBudgetMiddleware(budget=2000)],
)

print("=== Invocación con presupuesto suficiente ===")
resp = agent_budget.invoke(
    {
        "messages": [HumanMessage(content="¿Qué es un transformer en IA?")],
        "token_count": 0,  # inicializar el campo personalizado
    }
)
print("Respuesta:", resp["messages"][-1].content)
print(f"Tokens finales en estado: {resp.get('token_count', 'N/A')}")

In [ ]:
# Agente con presupuesto MUY bajo para demostrar el corte
agent_budget_tight = create_agent(
    model=llm,
    system_prompt="Eres un asistente conciso. Responde en español.",
    middleware=[TokenBudgetMiddleware(budget=1)],
)

print("=== Invocación con presupuesto agotado desde el inicio ===")
resp_tight = agent_budget_tight.invoke(
    {
        "messages": [HumanMessage(content="¿Cuánto es 2+2?")],
        "token_count": 5,  # ya estamos 'sobre el presupuesto' de 1
    }
)
print("Respuesta:", resp_tight["messages"][-1].content)

---

## Hook 4 — `wrap_model_call` con retry (wrap-style)

Los hooks **wrap-style** reciben el `request` y un `handler` (la función que realmente llama al modelo). Tú decides **cuántas veces** llamar al handler:
- **0 veces** → short-circuit (retornar una respuesta sin llamar al LLM)
- **1 vez** → flujo normal
- **N veces** → retry logic

Esto es imposible con hooks node-style, que solo observan el estado antes/después de que la infraestructura llame al modelo.

En este ejemplo implementamos **retry con backoff exponencial**.

In [ ]:
import asyncio
import time
from typing import Awaitable, Callable

from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse


class RetryMiddleware(AgentMiddleware):
    """Reintenta la llamada al modelo ante errores transitorios.

    Hooks utilizados:
      - wrap_model_call  → versión sync
      - awrap_model_call → versión async (para ainvoke)

    Parámetros:
      max_retries: número máximo de intentos (default: 3)
      base_delay:  tiempo base entre reintentos en segundos (default: 1.0)
                   El delay real es base_delay * 2^intento (backoff exponencial)
    """

    def __init__(self, max_retries: int = 3, base_delay: float = 1.0) -> None:
        super().__init__()
        self.max_retries = max_retries
        self.base_delay = base_delay

    # ── Versión SYNC ────────────────────────────────────────────────────────

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        last_error: Exception | None = None

        for attempt in range(self.max_retries):
            try:
                if attempt > 0:
                    print(f"[RetryMiddleware] Intento {attempt + 1}/{self.max_retries}")
                result = handler(request)  # llama al modelo
                if attempt > 0:
                    print(f"[RetryMiddleware]  Éxito en intento {attempt + 1}")
                return result
            except Exception as e:
                last_error = e
                if attempt < self.max_retries - 1:
                    delay = self.base_delay * (2 ** attempt)
                    print(f"[RetryMiddleware]   Error: {e}. Reintentando en {delay:.1f}s...")
                    time.sleep(delay)

        raise RuntimeError(
            f"[RetryMiddleware] Fallaron {self.max_retries} intentos. "
            f"Último error: {last_error}"
        )

    # ── Versión ASYNC ───────────────────────────────────────────────────────

    async def awrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], Awaitable[ModelResponse]],
    ) -> ModelResponse:
        last_error: Exception | None = None

        for attempt in range(self.max_retries):
            try:
                if attempt > 0:
                    print(f"[RetryMiddleware] Intento async {attempt + 1}/{self.max_retries}")
                result = await handler(request)  # llama al modelo de forma async
                if attempt > 0:
                    print(f"[RetryMiddleware]  Éxito en intento {attempt + 1}")
                return result
            except Exception as e:
                last_error = e
                if attempt < self.max_retries - 1:
                    delay = self.base_delay * (2 ** attempt)
                    print(f"[RetryMiddleware]   Error: {e}. Reintentando en {delay:.1f}s...")
                    await asyncio.sleep(delay)

        raise RuntimeError(
            f"[RetryMiddleware] Fallaron {self.max_retries} intentos async. "
            f"Último error: {last_error}"
        )


print("RetryMiddleware definido ✓")

In [ ]:
# En condiciones normales el retry no se activa, pero el middleware está listo
# para cuando haya errores de rate limit o timeout
agent_retry = create_agent(
    model=llm,
    system_prompt="Eres un asistente conciso. Responde en español.",
    middleware=[RetryMiddleware(max_retries=3, base_delay=0.5)],
)

resp = agent_retry.invoke(
    {"messages": [HumanMessage(content="¿Cuántos planetas tiene el sistema solar?")]}
)
print("Respuesta:", resp["messages"][-1].content)

---

## Hook 5 — `before_agent` + `after_agent` (ciclo de vida completo)

Los hooks `before_agent` y `after_agent` se ejecutan **una sola vez** por invocación (no en cada iteración del loop). Son ideales para:
- Inicializar recursos o registrar el inicio de una sesión
- Limpiar recursos o registrar métricas de la sesión completa al finalizar

In [ ]:
import time
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState
from langgraph.runtime import Runtime


class SessionTrackingMiddleware(AgentMiddleware):
    """Registra el inicio y fin de cada invocación del agente.

    Hooks utilizados:
      - before_agent → registra inicio de sesión
      - after_agent  → registra fin de sesión y métricas totales
    """

    _session_start: float = 0.0

    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        self._session_start = time.perf_counter()
        user_msg = state["messages"][0].content if state["messages"] else "(vacío)"
        print(f"[SessionTracking] Sesión iniciada")
        print(f"[SessionTracking]    Primera pregunta: '{user_msg[:60]}...' ")
        return None

    def after_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        total_ms = (time.perf_counter() - self._session_start) * 1000
        total_msgs = len(state["messages"])
        # Sumar tokens de todos los mensajes del agente
        total_tokens = sum(
            (getattr(m, "usage_metadata", None) or {}).get("output_tokens", 0)
            for m in state["messages"]
        )
        print(f"[SessionTracking] Sesión finalizada")
        print(f"[SessionTracking]    Tiempo total: {total_ms:.0f}ms")
        print(f"[SessionTracking]    Mensajes generados: {total_msgs}")
        print(f"[SessionTracking]    Tokens de salida totales: {total_tokens}")
        return None


print("SessionTrackingMiddleware definido ✓")

In [ ]:
agent_session = create_agent(
    model=llm,
    system_prompt="Eres un asistente conciso. Responde en español.",
    middleware=[SessionTrackingMiddleware()],
)

resp = agent_session.invoke(
    {"messages": [HumanMessage(content="Explica brevemente qué es el aprendizaje profundo.")]}
)
print("\nRespuesta:", resp["messages"][-1].content)

---

## Composición: todos los middlewares juntos

Los middlewares se **apilan** en la lista. El orden determina cómo se ejecutan:

```python
middleware=[
    SessionTrackingMiddleware(),   # 1° before_agent / último after_agent
    LoggingMiddleware(),           # 2° before_model / 2°-último after_model
    TokenBudgetMiddleware(2000),   # 3° before_model (verifica) / after_model (acumula)
    SafetyGuardMiddleware(),       # 4° after_model (inspecciona)
    RetryMiddleware(max_retries=2),# 5° wrap_model_call (más interno)
]
```

Diagrama de ejecución:
```
SessionTracking.before_agent
  LoggingMiddleware.before_model
    TokenBudget.before_model     ← verifica presupuesto
      RetryMiddleware.wrap_model_call [
        → llama al modelo
      ]
    SafetyGuard.after_model      ← inspecciona respuesta
    TokenBudget.after_model      ← acumula tokens
  LoggingMiddleware.after_model
SessionTracking.after_agent
```

In [ ]:
agent_full = create_agent(
    model=llm,
    system_prompt="Eres un asistente conciso. Responde en español.",
    middleware=[
        SessionTrackingMiddleware(),
        LoggingMiddleware(),
        TokenBudgetMiddleware(budget=2000),
        SafetyGuardMiddleware(),
        RetryMiddleware(max_retries=2, base_delay=0.5),
    ],
)

print("=" * 60)
resp_full = agent_full.invoke(
    {
        "messages": [HumanMessage(content="¿Qué es RAG en el contexto de LLMs?")],
        "token_count": 0,  # campo personalizado de TokenBudgetState
    }
)
print("=" * 60)
print("\nRespuesta final:", resp_full["messages"][-1].content)

---

## Resumen: cuándo usar cada hook

| Hook | Tipo | Cuándo usarlo | Ejemplo en este notebook |
|------|------|--------------|-------------------------|
| `before_agent` | node-style | Inicializar sesión, cargar contexto global | `SessionTrackingMiddleware` |
| `before_model` | node-style | Validar estado, verificar límites, logear entrada | `LoggingMiddleware`, `TokenBudgetMiddleware` |
| `after_model` | node-style | Inspeccionar respuesta, acumular métricas, safety checks | `LoggingMiddleware`, `SafetyGuardMiddleware`, `TokenBudgetMiddleware` |
| `after_agent` | node-style | Limpiar recursos, reportar métricas de sesión | `SessionTrackingMiddleware` |
| `wrap_model_call` | wrap-style | Retries, caché, modificar request/response, short-circuit | `RetryMiddleware`, `SkillMiddleware` |
| `wrap_tool_call` | wrap-style | Auditoría de tools, transformación de resultados, timeouts | *(no demostrado aquí)* |